# Model data

Geological model data typically consists of structured, multidimensional data representing two types of models: **voxel models** and **layer models**. These models are often stored in NetCDF format, which can be handled efficiently using the popular Python library [Xarray](https://docs.xarray.dev/en/stable/).

GeoST [extends Xarray](https://docs.xarray.dev/en/stable/internals/extending-xarray.html) with a 
`.gst` accessor that provides GeoST-specific functionality for Xarray's core `Dataset` and `DataArray` objects. The accessor becomes available when GeoST is imported as follows:

In [ ]:
import geost

Once imported, the `.gst` accessor can be used on `Dataset` and `DataArray` objects. In general, voxel models and layer models can be distinguished based on their dimensions:

- voxelmodel: `"x"`, `"y"`, and `"z"` dimensions
- layermodel: `"x"`, `"y"`, and `"layer"` dimensions, with `"top"` and `"bottom"` information for each `"layer"`

For the accessor to work, all three dimensions required for either a voxel model or a layer model must be present. Note that the distinction between a voxel model and a layer model is determined by the third dimension: `"z"` or `"layer"`. This dimension determines whether the data is handled as a voxel model or a layer model.

The dimension names listed above are not required to match exactly, similar to the [positional columns](survey_data.ipynb#postional_columns) of survey data. For example, latitude and longitude coordinates are also valid `"x"` and `"y"` dimensions. The table below shows the possible names for each dimension that are recognized by the accessor.

| Dimension | Model type | Names
| - | - | - |
| x | voxel/layer | "x", "xco", "xcoord", "longitude", "lon", "easting" |
| y | voxel/layer | "y", "yco", "ycoord", "latitude", "lat", "northing" |
| z | voxel | "z", "depth", "elevation" |
| z | layer | "layer", "unit", "horizon", "stratunit" |
| top | layer | "top", "tv_top_nap", "top_diepte", "top_depth", "upperboundary" |
| bottom | layer | "bottom", "tv_bottom_nap", "basis_diepte", "bottom_depth", "lowerboundary" |

```{note}
The third dimension in both a voxel model and a layer model is referred to as the z-dimension. In a voxel model, the z-dimension generally represents depth or elevation with respect to NAP. In a layer model, it generally represents a unit (e.g. a stratigraphic unit), while the associated depths are stored in `"top"` and `"bottom"` variables.

In [ ]:
geotop = geost.data.geotop_usp()
geotop

Note that GeoTOP is an `xarray.Dataset` instance. In the coordinates section, we can see the names of the dimensions: "x", "y", and "z". We can verify that these dimensions are indeed recognized by GeoST:

In [ ]:
print(f"x-dimension: {geotop.gst.x_dim}")
print(f"y-dimension: {geotop.gst.y_dim}")
print(f"z-dimension: {geotop.gst.z_dim}")

Attempting to call the accessor when not all required dimensions can be found will raise an error. An error will also be raised when the z-dimension of a `Dataset` or `DataArray` is ambiguous, meaning that it could represent either a voxel model or a layer model: for example, the `Dataset` contains both a "z" and "layer" dimension.

In [ ]:
import numpy as np
import xarray as xr

# Create an example Dataset with "z" and "layer" dimensions.
invalid_model = xr.Dataset(
    data_vars={"data": (("y", "x", "z", "layer"), np.zeros((3, 3, 2, 1)))},
    coords={"x": [0, 1, 2], "y": [2, 1, 0], "z": [0, 1], "layer": [0]},
)
try:
    invalid_model.gst
except geost.exceptions.InvalidModelError as e:
    print(e)

## Working with model data

We will continue this example with the previously loaded GeoTOP data for the Utrecht Science
Park. As said, all GeoST functionality is available through the `.gst` accessor. For example, we can check the horizontal and vertical bounds of the GeoTOP data:

In [ ]:
print(geotop.gst.bounds())
print(geotop.gst.vertical_bounds)

This prints the "xmin, ymin, xmax, ymax" bounding box and the "zmin, zmax" of the data. Because
we only extend Xarray objects, familiar selection methods `.sel` and `.isel`, to select specific coordinates or indices, can be used directly.

In [ ]:
sel = geotop.sel(x=[139_650, 139_750])  # Select specific x-coordinates
sel = geotop.isel(x=[2, 5])  # Select specific x-indices

sel = geotop.sel(x=slice(140_000, 140_500))  # Select a slice of x-coordinates
sel = geotop.isel(x=slice(2, 5))  # Select a slice of x-indices

sel = geotop.sel(
    x=[140_013.23, 140_333.14], method="nearest"
)  # Select the nearest x-coordinates
print(sel)

Using GeoST, we can select within a bounding box very easily.

In [ ]:
xmin, ymin, xmax, ymax = 140_000, 455_500, 141_000, 456_000
selection = geotop.gst.select_within_bbox(xmin, ymin, xmax, ymax)

If your coordinates are in a different CRS, it also works by specifying the CRS of your
bounding box coordinates.

In [ ]:
from geost.utils import projections

# Transform coordinates from EPSG:28992 (Amersfoort / RD New) to EPSG:32631 (WGS 84 / UTM zone 31N)
transformer = projections.horizontal_reference_transformer(28992, 32631)
utm_xmin, utm_ymin = transformer.transform(xmin, ymin)
utm_xmax, utm_ymax = transformer.transform(xmax, ymax)

selection = geotop.gst.select_within_bbox(
    utm_xmin, utm_ymin, utm_xmax, utm_ymax, crs=32631
)

## Overview of attributes and methods

Many attributes and methods are implemented in the `.gst` accessor for analyses and spatial selections besides the ones demonstrated above. A short summary of the available attributes and methods is presented here. A complete overview can be found in the [ModelDataArray](../api_reference/model_dataarray.rst) and [ModelDataset](../api_reference/model_dataset.rst) sections of the API reference.

All methods listed below work with both voxel model and layer model `Dataset`/`DataArray` objects.

**Attributes**
- `x`: x-coordinates of the model
- `y`: y-coordinates of the model
- `z`: z-coordinates of the model
- `crs`: Coordinate reference system (CRS) of the model
- `resolution`: Resolution of the model: (x,y,z)-resolution for a voxelmodel and (x,y)-resolution for a layermodel
- `bounds`: xmin, ymin, xmax, ymax bounding box of the model
- `vertical_bounds`: Depth min, depth max bounds of the model
- `surface_level`: 2D grid of the surface level of the voxel- or layermodel model

**Analysis/selection**
- `get_thickness`: Calculate the thickness where a model matches a condition
- `most_common`: Determine the most common value (mode) in a model
- `slice_depth_interval`: Slice a specific depth interval from a model

**Spatial methods**
- `select_within_bbox`: Select data within an xmin, ymin, xmax, ymax bounding box from the model
- `mask_geometries`: Mask specific cells that overlap with point/line/polygon geometries in the model
- `select_points`: Select specific locations from the model



## Reading model data

GeoST provides several readers for general and specific model data, such as GeoTOP and REGIS II. These return Xarray objects. Modeldata can be read from local NetCDF files or from OPeNDAP
services. The table below lists the different readers for the different types of models.

| Model | Read function | Returns | Description |
| - | - | - | - |
| Generic modeldata | [`read_model_netcdf`](../api_reference/generated/geost.read_model_netcdf.rst) [`read_model_from_opendap`](../api_reference/generated/geost.read_model_from_opendap.rst) | `xarray.Dataset` or `xarray.DataArray` | Generic reader for any kind of modeldata available in NetCDF format or from an OPeNDAP service |
| GeoTOP (BRO) | [`read_geotop_netcdf`](../api_reference/generated/geost.read_geotop_netcdf.rst) [`read_geotop_from_opendap`](../api_reference/generated/geost.read_geotop_from_opendap.rst) | `xarray.Dataset` or `xarray.DataArray` | Reader for [GeoTOP](https://basisregistratieondergrond.nl/inhoud-bro/registratieobjecten/modellen/geotop-gtm/) from a local NetCDF or the [Dinodata OPeNDAP service](https://www.dinodata.nl/opendap/) |
| REGIS II (BRO) | [`read_regis_netcdf`](../api_reference/generated/geost.read_regis_netcdf.rst) [`read_regis_from_opendap`](../api_reference/generated/geost.read_regis_from_opendap.rst) | `xarray.Dataset` or `xarray.DataArray` | Reader for [REGIS II](https://basisregistratieondergrond.nl/inhoud-bro/registratieobjecten/modellen/regis-ii-hydrogeologisch-model-hgm/) from a local NetCDF or the [Dinodata OPeNDAP service](https://www.dinodata.nl/opendap/) |